In [1]:
import socket
import threading

# TESTING_HOSTNAME = 'localhost'
# TESTING_PORT = 12000

BUFFER_SIZE = 4096

In [2]:

def default_on_recv(data_raw, addr):
    data = data_raw.decode()
    
    print(f"From: {addr[0]} / {addr[1]}")
    print(f"Data: {data}")

# def udp_stream(host, port, on_recv=default_on_recv, buffer_size=4096, timeout=1.0):
#     """
#     Receives UDP packets and calls on_frame(data) for each valid JPEG frame.
#     Returns a stop function — call it to end the stream.
#     """
#     stop_event = threading.Event()
#     sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
#     sock.bind((host, port))
#     sock.settimeout(timeout)

#     def loop():
#         while not stop_event.is_set():
#             try:
#                 data, addr = sock.recvfrom(buffer_size)
#                 on_recv(data, addr)
#             except socket.timeout:
#                 continue
#         sock.close()

#     thread = threading.Thread(target=loop, daemon=True)
#     thread.start()

#     return stop_event.set  # caller gets back just the stop function

class UDPStream:
    def __init__(self, port: int, dest_host: str = "", on_recv: callable = default_on_recv, buffer_size: int = 4096, timeout: float = 1.0):
        self._stop_event = threading.Event()
        self._sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
        self._sock.bind(('', port))
        self._sock.settimeout(timeout)

        self._default_dest_host = dest_host
        self._port = port
        self._buffer_size = buffer_size
        self._on_recv = on_recv

        thread = threading.Thread(target=self._loop, daemon=True)
        thread.start()

    def __del__(self):
        self.stop()

    def _loop(self):
        while not self._stop_event.is_set():
            try:
                data, addr = self._sock.recvfrom(self._buffer_size)
                self._on_recv(data, addr)
            except socket.timeout:
                continue
        self._sock.close()

    def send(self, data, host=None):
        if host is None: host = self._default_dest_host
        self._sock.sendto(data, (host, self._port))

    def stop(self):
        self._stop_event.set()

In [9]:
# import ipywidgets as widgets
# from IPython.display import display

# img_widget = widgets.Image(format='jpeg')
# img_widget.layout.width = '10%' 
# display(img_widget)

SERVER_PORT = 1901

def on_recv_broadcast(data_raw, addr):
    print(data_raw[1:4])
    if data_raw[1:4] == b'\xff\xd8\xff':
        # setattr(img_widget, 'value', data_raw[1:])
        pass
    else:
        data = data_raw.decode()
        print(data)
    
    print(f"From: {addr[0]} / {addr[1]}")

broadcast_listener = UDPStream(SERVER_PORT, on_recv=on_recv_broadcast, buffer_size=10000)


In [10]:
broadcast_listener.stop()

In [4]:
def calcCheckSum(msg: str) -> str:
    check_sum = sum(ord(chr) for chr in msg) % 0x100
    # check_sum = sum % 0xff
    return f"{check_sum:X}"

def prepForSend(msg: str) -> str:
    return f"[{msg}{calcCheckSum(msg)}]"

In [5]:
# Had to give myself 192.168.200.x subnet alias as kairos is there and can't route to 192.168.0.x I guess
# Command I used was
# sudo ifconfig en0 alias 192.168.200.50 255.255.255.0

KAIROS_HOST = "192.168.200.220"

SHARED_LINK_OUTBOUND_PORT = 4000
SHARED_LINK_INBOUND_PORT = 4220

MY_HOST = "192.168.200.50"

declare_myIP_msg = f":BA|ELink|D0, SetIP, {MY_HOST}:{SHARED_LINK_INBOUND_PORT}, 4000|C"

enab_msgs = [
    ":BA|ELink|D0, outbound, on|C",
    ":BA|ELink|D0, inbound, on|C"
]

list_SVs_msgs = [
    ":BA|MUI|C",
    ":BA|MUO|C",
    ":BA|N0|T0|Qvideo_mux1|T0|Qveh_start|T0|Qveh_enable|T0|Qveh_motion|T2|Qjoy_steer|T0|Qveh_steer|T0|Qveh_brake|T0|Qveh_throttle|T0|Qveh_shift|T0|Qveh_clutch|C",
    ":BA|L0|T2|QAutoGpsHead|T2|QAutoGpsLat|T2|QAutoGpsLon|T2|QAutoGpsVel|T0|QAutoGpsReferences|T0|QAutoGpsQuality|T2|QAutoGpsStamp|T0|Qpause|T0|Qkey_on|T0|Qeng_enb|T0|Qpe7|T0|Qbbrake|T0|Qfbrake|T2|Qsteerang|T0|Qthrottle|Qsvp_enb_eng|T0|Qserialpck|T0|Qveh_gear|T0|Qveh_state|T0|Qveh_estop|T0|Qveh_obstacle|T2|Qvehicle_bat|T4|Qrun_composite|T4|QVehicleName|T0|QHealth_total|T0|Qrpm|T0|Qfuel|T2|Qsys_thd|C"
    ":BA|MW|C",
    ":BA|ML|C"
]

# Should respond with something like
# [:AB|N0|Rvideo_mux1|D0|Rveh_start|D0|Rveh_enable|D0|Rveh_motion|D0|Rjoy_steer|D0|Rveh_steer|D0|Rveh_brake|D100|Rveh_throttle|D0|Rveh_shift|D0|Rveh_clutch|D0|C92]
# Not sure if it'll also have that for the quires for KAIROS's outbound variables too 
# this might even be unnecessary and just used to confirm that it is getting the changes

term_msgs = [
    ":BA|ELink|D0, outbound, off|C",
    ":BA|ELink|D0, inbound, off|C"
]

ping_msg = ":BA|P0|C"
pong_msg = ":BA|G0|C"

teleop_start = ":BA|EServoPod|D0,TELEOP START|C"
teleop_stop = ":BA|EServoPod|D0,TELEOP STOP|C"

In [27]:
from collections import OrderedDict
from dataclasses import dataclass
from typing import Union

from UDPConnection import UdpConnection
from premade_msgs import *

@dataclass
class SharedVariable:
    type: int  # 0=long, 2=double, 4=string
    value: Union[int, float, str, None] = None
    updated: bool = False

    def setValue(self, val):
        self.value = val
        self.updated = True

class SharedLink:
    def __init__(self):
        self._outbound_SVs = self._parse_sv_list(list_SVs_msgs[2])
        self._inbound_SVs = self._parse_sv_list(list_SVs_msgs[3])
        self._udp = UdpConnection(KAIROS_IP, KAIROS_PORT, LOCAL_PORT, self._msg_parser)

        self._inbound_data_counter = 0

    def __del__(self):
        del self._udp

    def stop(self):
        self._udp.stop()

    def _msg_parser(self, data_raw, addr):
        # was getting error that it couldn't decode xff, but it also wasn't 
        if data_raw[1:4] == b'\xff\xd8\xff':
            # setattr(img_widget, 'value', data_raw[1:])
            print("Seems to be a JPEG")
        else:
            data = data_raw.decode("utf-8")

            if data[:5] != "[:AB|":
                print("Data isn't in SharedLink form")
                print(data)
                return
            
            split_data = data[1:-1].split("|")

            # assume that checksum is always right because sent over UDP
            # assert calcCheckSum(data[1:-3]) == data[-3:-1]

            match split_data[1]:
                case "N0":        
                    match split_data[2][0]:
                        case "D":
                            self._apply_inbound_data(split_data[2:-1])
                        case "R":
                            print("Received R type:")
                            print(data)
                        case _:
                            print(f"Received N0 then {split_data[2]}")
                            print(data)
                case "P0":
                    # print("Received Ping Request")
                    self.sendMsg(pong_msg)
                case "G0":
                    print("Received Pong Message (ping ACK)")
                case _:
                    print(f"Received {split_data[1]} msg")
                
        
        # print(f"From: {addr[0]} / {addr[1]}")

    def _apply_inbound_data(self, data_values: list):
        """Apply a list of D field values to the inbound map in order."""
        assert len(self._inbound_SVs) == len(data_values)

        for (name, sv), value in zip(self._inbound_SVs.items(), data_values):
            assert value[0] == "D"
            value = value[1:]

            if value is not None and value != '':
                # Cast to correct type
                if sv.type == 0:
                    sv.value = int(value)
                elif sv.type == 2:
                    sv.value = float(value)
                elif sv.type == 4:
                    sv.value = str(value)
                sv.updated = False

        self._inbound_data_counter += 1
        if self._inbound_data_counter % 50 == 49:
            print(f"50 inbound data received: #{self._inbound_data_counter // 50}") 

    def _parse_sv_list(self, command: str) -> OrderedDict:
        """Parse a SharedLink list command into an ordered dict of SharedVariables."""
        result = OrderedDict()
        
        # Strip the framing and checksum
        # Remove leading/trailing whitespace and brackets
        inner = command.strip().lstrip(':')
        
        # Split on pipes
        fields = inner.split('|')
        
        current_type = 0  # default to long if no type specified
        for field in fields:
            if field.startswith('T'):
                # Type field - next variable will use this type
                try:
                    current_type = int(field[1:])
                except ValueError:
                    pass
            elif field.startswith('Q'):
                # Variable name
                name = field[1:]
                result[name] = SharedVariable(type=current_type)
                current_type = 0  # reset to default after each variable
        
        return result
    
    def printData(self, outbound=True, inbound=False):
        print("Outbound data:")
        for name, sv in self._outbound_SVs.items():
            print(f"{name}: type={sv.type}, value={sv.value}, updated={sv.updated}")

        print("\nInbound data:")
        for name, sv in self._inbound_SVs.items():
            print(f"{name}: type={sv.type}, value={sv.value}, updated={sv.updated}")
    
    def setData(self, name: str, value: Union[int, float, str]):
        assert name in self._outbound_SVs

        self._outbound_SVs[name].setValue(value)

    def sendData(self):
        msg = ":BA|N0|"
        for name, sv in self._outbound_SVs.items():
            msg += f"D{sv.value}|"
        msg += "C"
        self.sendMsg(msg)
        

    def sendMsg(self, msg: str):
        self._udp.send(prepForSend(msg).encode("utf-8"))

    def sendMsgs(self, msgs: list):
        for msg in msgs:
            self._udp.send(prepForSend(msg).encode("utf-8"))


In [25]:
talker.stop()

In [23]:
talker = SharedLink()

In [56]:
talker.setData("video_mux1", 0)
talker.setData("veh_start", 0)
####
talker.setData("veh_enable", 1)

talker.setData("veh_motion", 0)

#####
talker.setData("joy_steer", 0)

talker.setData("veh_steer", 0)

#####
talker.setData("veh_brake", 0)
talker.setData("veh_throttle", 100)

talker.setData("veh_shift", 0)
talker.setData("veh_clutch", 0)

In [57]:
talker.sendData()

In [59]:
# talker.sendMsg(declare_myIP_msg)
# talker.sendMsgs(enab_msgs)
# talker.sendMsgs(list_SVs_msgs)
# talker.sendMsg(list_SVs_msgs[2])
# talker.sendMsg(ping_msg)
# talker.sendMsg(teleop_start)
talker.sendMsgs(term_msgs)

In [58]:
talker.printData()

Outbound data:
video_mux1: type=0, value=0, updated=True
veh_start: type=0, value=0, updated=True
veh_enable: type=0, value=1, updated=True
veh_motion: type=0, value=0, updated=True
joy_steer: type=2, value=0, updated=True
veh_steer: type=0, value=0, updated=True
veh_brake: type=0, value=0, updated=True
veh_throttle: type=0, value=100, updated=True
veh_shift: type=0, value=0, updated=True
veh_clutch: type=0, value=0, updated=True

Inbound data:
AutoGpsHead: type=2, value=None, updated=False
AutoGpsLat: type=2, value=0.0, updated=False
AutoGpsLon: type=2, value=0.0, updated=False
AutoGpsVel: type=2, value=0.0, updated=False
AutoGpsReferences: type=0, value=0, updated=False
AutoGpsQuality: type=0, value=0, updated=False
AutoGpsStamp: type=2, value=6014.265, updated=False
pause: type=0, value=0, updated=False
key_on: type=0, value=1, updated=False
eng_enb: type=0, value=1, updated=False
pe7: type=0, value=1, updated=False
bbrake: type=0, value=100, updated=False
fbrake: type=0, value=0, u

In [103]:
# MY_HOST = "192.168.0.215"
MY_HOST = "192.168.200.50"

msg = f":BA|ELink|D0, SetIP, {MY_HOST}:{SHARED_LINK_INBOUND_PORT}, 4000|C"
msg = f"[{msg}{calcCheckSum(msg)}]"

print(msg)

shared_link_outbound.send(msg.encode('utf-8'))

[:BA|ELink|D0, SetIP, 192.168.200.50:4220, 4000|CE6]


In [16]:



def on_recv_broadcast(data_raw, addr):
    # was getting error that it couldn't decode xff, but it also wasn't 
    if data_raw[1:4] == b'\xff\xd8\xff':
        # setattr(img_widget, 'value', data_raw[1:])
        print("Seems to be a JPEG")
    else:
        data = data_raw.decode("utf-8")

        if data[:5] != "[:AB|":
            print("Data isn't in SharedLink form")
            print(data)
            return
        
        split_data = data[1:-1].split("|")

        if split_data[1] == "N0":
            pass

        print(data)
    
    print(f"From: {addr[0]} / {addr[1]}")

shared_link_outbound = UDPStream(SHARED_LINK_OUTBOUND_PORT, KAIROS_HOST, on_recv_broadcast)
shared_link_inbound = UDPStream(SHARED_LINK_INBOUND_PORT, "", on_recv_broadcast)

OSError: [Errno 98] Address already in use

In [26]:
shared_link_outbound.stop()
shared_link_inbound.stop()

NameError: name 'shared_link_outbound' is not defined

||SN23030555|192.168.200.220|VEH_NorthEastern1|Vehicle_SH/SL/IVN/DWB|djBasis.exe, djJAUS_IVN.exe, djEtherMap.exe, DJ_LDR.EXE, djSharedLinkF.exe, GPS2RDDF.EXE, djMimic.exe, djPinger.exe, djDriveWB.exe|0|0|0|0|||01-01-2012|00:00:33|SharedLink,192.168.200.220:4000:4220;Cmd: Restart System,restart djBasis,7101;Cmd: Guideline dist,follow #,7101||

In [1]:
f"wow{None}hiii"

'wowNonehiii'